# Abstractive Summarization — T5

T5 (Text-to-Text): semua tugas diperlakukan sebagai teks-ke-teks. Versi ini sudah di-fine-tune khusus untuk meringkas bahasa Indonesia.

Input: `data_final_preprocessing.csv` (sudah dipreprocessing, tidak ada preprocessing lagi di sini).

**Struktur:** Load → Muat Model → Fungsi → ROUGE → BERTScore → Contoh → Simpan.

---
## Setup

In [1]:
!pip install transformers sentencepiece rouge-score bert-score -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26

In [2]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize
from transformers import T5ForConditionalGeneration, T5Tokenizer
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from tqdm import tqdm
tqdm.pandas()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


---
## 1. Load Dataset

In [3]:
df = pd.read_csv('/kaggle/input/datasets/nazhifberlian/nlp-wikipedia-summarization/data_final_preprocessing.csv',
                 encoding='utf-8-sig')

print(f'Dataset dimuat: {len(df)} baris, {len(df.columns)} kolom')
print(f'Kolom: {list(df.columns)}')
print(f'\nDistribusi kategori:')
print(df['category'].value_counts().to_string())

Dataset dimuat: 12644 baris, 9 kolom
Kolom: ['global_id', 'id', 'title', 'category', 'article_text', 'body_word_count', 'lead_paragraph', 'lead_word_count', 'sentences']

Distribusi kategori:
category
sejarah     1976
arts        1962
artis       1947
kuliner     1947
tech        1716
biografi    1679
sains       1417


---
## 2. Muat Model
Pada abstractive, tahap ini memuat model generatif (encoder-decoder), menggantikan langkah stopword pada extractive.

In [4]:
MODEL_NAME = 'cahya/t5-base-indonesian-summarization-cased'

print(f'Memuat model {MODEL_NAME}...')
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print(f'Model siap | Parameter: {model.num_parameters():,}')

Memuat model cahya/t5-base-indonesian-summarization-cased...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/793k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/657 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model siap | Parameter: 296,926,464


---
## 3. Fungsi Summarization
Termasuk chunking untuk artikel panjang agar muat di batas token model.

In [5]:
MAX_INPUT_TOKENS  = 512
MAX_OUTPUT_TOKENS = 128
MIN_OUTPUT_TOKENS = 30
MAX_SENTENCES_PER_CHUNK = 20

def split_sentences(text):
    return [s.strip() for s in sent_tokenize(str(text)) if len(s.split()) >= 4]

def summarize_transformer(text):
    inputs = tokenizer(str(text), return_tensors='pt',
                       max_length=MAX_INPUT_TOKENS, truncation=True, padding=False).to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_OUTPUT_TOKENS,
            min_new_tokens=MIN_OUTPUT_TOKENS,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=1.0)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

def summarize_with_chunking(text):
    # Artikel panjang dipecah agar muat di batas token model (khusus transformer).
    sentences = split_sentences(text)
    if len(sentences) <= MAX_SENTENCES_PER_CHUNK:
        return summarize_transformer(text)
    chunks = [sentences[i:i+MAX_SENTENCES_PER_CHUNK]
              for i in range(0, len(sentences), MAX_SENTENCES_PER_CHUNK)]
    chunk_summaries = [summarize_transformer(' '.join(c)) for c in chunks]
    return summarize_transformer(' '.join(chunk_summaries))

print('Fungsi summarization siap.')

Fungsi summarization siap.


### Generate ringkasan untuk seluruh dataset

In [6]:
print('Generating ringkasan T5...')
start = time.time()

df['summary_t5'] = df['article_text'].progress_apply(summarize_with_chunking)

time_gen = time.time() - start
print(f'Selesai dalam {time_gen:.1f} detik ({time_gen/len(df):.2f} detik/artikel)')

Generating ringkasan T5...



100%|██████████| 12644/12644 [10:07:23<00:00,  2.88s/it]

Selesai dalam 36443.2 detik (2.88 detik/artikel)


---
## 4. Evaluasi ROUGE

In [7]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

r1, r2, rl = [], [], []
for pred, ref in zip(df['summary_t5'], df['lead_paragraph']):
    s = scorer.score(str(ref), str(pred))
    r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure)
    rl.append(s['rougeL'].fmeasure)

rouge_result = (np.mean(r1), np.mean(r2), np.mean(rl))

print('Evaluasi ROUGE — T5')
print(f'{"Metrik":<10} | {"Skor":>8}')
print('-' * 22)
print(f'{"ROUGE-1":<10} | {rouge_result[0]:>8.4f}')
print(f'{"ROUGE-2":<10} | {rouge_result[1]:>8.4f}')
print(f'{"ROUGE-L":<10} | {rouge_result[2]:>8.4f}')

Evaluasi ROUGE — T5
Metrik     |     Skor
----------------------
ROUGE-1    |   0.1723
ROUGE-2    |   0.0395
ROUGE-L    |   0.1156


### ROUGE-L per Kategori

In [8]:
rows = []
for kat in sorted(df['category'].unique()):
    sub = df[df['category'] == kat]
    scores = [scorer.score(str(ref), str(pred))['rougeL'].fmeasure
              for pred, ref in zip(sub['summary_t5'], sub['lead_paragraph'])]
    rows.append({'kategori': kat, 'jumlah': len(sub), 'rougeL': np.mean(scores)})

hasil_kategori = pd.DataFrame(rows).sort_values('rougeL', ascending=False)
print('ROUGE-L per kategori (T5):')
print(hasil_kategori.round(4).to_string(index=False))

ROUGE-L per kategori (T5):
kategori  jumlah  rougeL
 sejarah    1976  0.1216
biografi    1679  0.1188
 kuliner    1947  0.1185
    tech    1716  0.1147
    arts    1962  0.1133
   artis    1947  0.1127
   sains    1417  0.1075


---
## 5. Evaluasi BERTScore
* BERTScore mengukur kemiripan makna (penting untuk abstractive yang memparafrase).
* Berat, jadi pakai sampel. lang='id' otomatis pakai model multilingual.

In [9]:
SAMPLE_SIZE_BS = 500
sample_bs = df.sample(n=min(SAMPLE_SIZE_BS, len(df)), random_state=42).reset_index(drop=True)

cands = sample_bs['summary_t5'].astype(str).tolist()
refs  = sample_bs['lead_paragraph'].astype(str).tolist()

P, R, F1 = bertscore(cands, refs, lang='id', verbose=False)
bertscore_result = (P.mean().item(), R.mean().item(), F1.mean().item())

print(f'Evaluasi BERTScore — T5 (sampel N={len(sample_bs)})')
print(f'{"Metrik":<12} | {"Skor":>8}')
print('-' * 24)
print(f'{"Precision":<12} | {bertscore_result[0]:>8.4f}')
print(f'{"Recall":<12} | {bertscore_result[1]:>8.4f}')
print(f'{"F1":<12} | {bertscore_result[2]:>8.4f}')

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluasi BERTScore — T5 (sampel N=500)
Metrik       |     Skor
------------------------
Precision    |   0.6809
Recall       |   0.6465
F1           |   0.6627


---
## 6. Contoh Ringkasan Hasil

In [10]:
N_CONTOH = 3
sample_idx = df.sample(N_CONTOH, random_state=42).index

for i, idx in enumerate(sample_idx):
    row = df.loc[idx]
    print('=' * 70)
    print(f'Artikel {i+1}: [{row["category"]}] {row["title"]}')
    print('-' * 20)
    print(f'Ground Truth (lead_paragraph):')
    print(f'{str(row["lead_paragraph"])[:200]}...')
    print('-' * 20)
    print(f'Ringkasan T5:')
    print(f'{str(row["summary_t5"])[:200]}')
    print()

Artikel 1: [sains] Atom Bakhidrogen
--------------------
Ground Truth (lead_paragraph):
suatu ion lirhidrogen bahasa inggris hydrogenlike ioncode en is deprecated  disebut pula ion mirip hidrogen merupakan inti atom yang memiliki satu elektron dan karenanya isoelektronik dengan hidrogen....
--------------------
Ringkasan T5:
Suatu orbital atom bakhidrogen secara unik diidentifikasi melalui nilai bilangan kuantum utama n, bilangan kuantum momentum sudut l, serta bilangan kuantum magnetik m. eigennilai energinya tidak berga

Artikel 2: [kuliner] Jeruk Bergamot
--------------------
Ground Truth (lead_paragraph):
jeruk bergamot adalah buah jeruk wangi dengan warna kuning atau hijau mirip dengan jeruk nipis, tergantung pada kematangannya. penelitian genetika tentang asal muasal leluhur kultivar jeruk yang masih...
--------------------
Ringkasan T5:
Bergamot adalah buah jeruk yang sebagian besar tumbuh di wilayah mediterania. produksinya dilakukan dalam skala besar di wilayah pesisir laut io

---
## 7. Simpan Hasil

In [11]:
output_cols = ['global_id', 'title', 'category', 'summary_t5',
               'lead_paragraph', 'body_word_count']
hasil = df[[c for c in output_cols if c in df.columns]]

hasil.to_csv('hasil_summary_t5.csv', index=False, encoding='utf-8-sig')
print(f'Tersimpan: hasil_summary_t5.csv ({len(hasil)} baris)')
print(f'Kolom: {list(hasil.columns)}')

Tersimpan: hasil_summary_t5.csv (12644 baris)
Kolom: ['global_id', 'title', 'category', 'summary_t5', 'lead_paragraph', 'body_word_count']
